In [1]:
# ── Librerías ─────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import janitor
import sqlalchemy as sa
import os
from pathlib import Path
from dotenv import load_dotenv

%matplotlib inline

# ── Opciones de visualización ─────────────────────────────────────────
# Desactivar notación científica
pd.set_option('display.float_format', '{:.3f}'.format)
np.set_printoptions(suppress=True)

# Ver todas las columnas al imprimir un DataFrame
pd.set_option('display.max_columns', None)

# ── Rutas del proyecto ────────────────────────────────────────────────
# Se resuelven desde la ubicación del notebook, así funcionan
# igual en cualquier máquina
RAIZ = Path.cwd().parent

ORIGINALES   = RAIZ / '02_datos' / '01_Originales'
VALIDACION   = RAIZ / '02_datos' / '02_Validacion'
ENTRENAMIENTO = RAIZ / '02_datos' / '03_Entrenamiento'
CACHES       = RAIZ / '02_datos' / '04_Caches'
MODELOS      = RAIZ / '05_modelos'
RESULTADOS   = RAIZ / '06_resultados'

# ── Variables de entorno ──────────────────────────────────────────────
load_dotenv(RAIZ / '.env')

print('✅ Entorno listo')
print(f'   Raíz del proyecto: {RAIZ}')

✅ Entorno listo
   Raíz del proyecto: c:\Users\Admin\Desktop\Desktop\CarpetaFisica\Data Detective\Ciencia_Datos_2\Caso_Agentes


In [2]:
# ── Carga del dataframe de trabajo ────────────────────────────────────
# Tablón procedente de la fase anterior (EDA), ver copilot-instructions.md
df = pd.read_pickle(ENTRENAMIENTO / '03_train_tablon_eda.pkl')

print('✅ Dataframe cargado')
print(f'   Filas:    {df.shape[0]:,}')
print(f'   Columnas: {df.shape[1]}')
df.shape, df.dtypes.head()

✅ Dataframe cargado
   Filas:    28,015
   Columnas: 16


((28015, 16),
 edad               int64
 trabajo         category
 estado_civil    category
 formacion       category
 impago          category
 dtype: object)

In [3]:
# ── FASE 1: Diagnóstico automático de variables ──────────────────────

# Tabla base de diagnóstico: tipo, cardinalidad y % de missing
diag = pd.DataFrame({
    'tipo_dato': df.dtypes.astype(str),
    'cardinalidad': df.nunique(),
    'pct_missing': (df.isnull().mean() * 100).round(2),
})

# Skew (asimetría) solo para columnas numéricas
num_cols = df.select_dtypes(include='number').columns
skew = df[num_cols].skew().round(2).rename('skew')
diag = diag.join(skew)

print('🔍 DIAGNÓSTICO GENERAL DE VARIABLES')
print('=' * 78)
print(diag.to_string())

# ── Distribución del target ───────────────────────────────────────────
print('\n🎯 DISTRIBUCIÓN DEL TARGET (contrata_fondos)')
print(df['contrata_fondos'].value_counts().to_string())
print(f'   % positivos: {df["contrata_fondos"].mean()*100:.2f}%')

# ── Categorías de cada variable categórica ────────────────────────────
print('\n🏷️ CATEGORÍAS DE VARIABLES CATEGÓRICAS')
for col in df.select_dtypes(include='category').columns:
    cats = df[col].cat.categories
    print(f'   • {col} ({len(cats)} categorías): {list(cats)[:12]}')

🔍 DIAGNÓSTICO GENERAL DE VARIABLES
                             tipo_dato  cardinalidad  pct_missing   skew
edad                             int64            78        0.000  0.830
trabajo                       category            12        0.000    NaN
estado_civil                  category             4        0.000    NaN
formacion                     category             8        0.000    NaN
impago                        category             3        0.000    NaN
prestamo_hipotecario          category             3        0.000    NaN
prestamo_personal             category             3        0.000    NaN
canal_de_contacto             category             2        0.000    NaN
mes                           category            10        0.000    NaN
num_contactos_esta_campana       int64            40        0.000  4.770
num_dias_ultimo_contacto         int64            26        0.000 -4.830
num_contactos_otras_campanas     int64             8        0.000  3.770
resultado_campan

In [8]:
# ── FASE 0: Identificación y marcado inicial ─────────────────────────
from sklearn.preprocessing import (OrdinalEncoder, OneHotEncoder,
                                   StandardScaler, FunctionTransformer)
from sklearn.impute import SimpleImputer

TARGET = 'contrata_fondos'

# Orden natural de formacion (de menor a mayor nivel educativo)
ORDEN_FORMACION = ['illiterate', 'basic.4y', 'basic.6y', 'basic.9y',
                   'high.school', 'professional.course', 'university.degree']
# Mapeo nombre de mes -> número (1..12) para codificación cíclica
MAP_MES = {'jan': 1, 'feb': 2, 'mar': 3, 'apr': 4, 'may': 5, 'jun': 6,
           'jul': 7, 'aug': 8, 'sep': 9, 'oct': 10, 'nov': 11, 'dec': 12}

# Listas de tracking del pipeline
cols_fase1_numericas = []   # Features numéricas a escalar en FASE 3
cols_fase2_noescalar = []   # Features binarias / cíclicas (no escalar)
cols_intermedias_excluir = []  # Originales/intermedias que NO van al df final

print('✅ FASE 0 OK')

# ── FASE 1: Features numéricas (a escalar en FASE 3) ─────────────────
df_f1 = pd.DataFrame(index=df.index)

# 1.1 formacion: OrdinalEncoding (unknown->NaN) + imputación mediana
oe_form = OrdinalEncoder(categories=[ORDEN_FORMACION],
                         handle_unknown='use_encoded_value', unknown_value=np.nan)
df_f1['formacion_oe'] = oe_form.fit_transform(df[['formacion']])[:, 0]
imp_form = SimpleImputer(strategy='median')
df_f1['formacion_oe_imp'] = imp_form.fit_transform(df_f1[['formacion_oe']])[:, 0]
cols_fase1_numericas.append('formacion_oe_imp')
cols_intermedias_excluir += ['formacion', 'formacion_oe']

# 1.2 num_dias_ultimo_contacto (Rama numérica): -1 -> NaN -> mediana
df_f1['num_dias_rec'] = df['num_dias_ultimo_contacto'].replace(-1, np.nan)
imp_dias = SimpleImputer(strategy='median')
df_f1['num_dias_ultimo_contacto_imp'] = imp_dias.fit_transform(df_f1[['num_dias_rec']])[:, 0]
cols_fase1_numericas.append('num_dias_ultimo_contacto_imp')
cols_intermedias_excluir += ['num_dias_ultimo_contacto', 'num_dias_rec']

# 1.3 Contactos a campañas: log1p (reduce cola larga, maneja el 0)
log1p_t = FunctionTransformer(func=np.log1p, validate=False)
df_f1['num_contactos_esta_campana_log'] = log1p_t.fit_transform(df[['num_contactos_esta_campana']]).to_numpy()[:, 0]
df_f1['num_contactos_otras_campanas_log'] = log1p_t.fit_transform(df[['num_contactos_otras_campanas']]).to_numpy()[:, 0]
cols_fase1_numericas += ['num_contactos_esta_campana_log', 'num_contactos_otras_campanas_log']
cols_intermedias_excluir += ['num_contactos_esta_campana', 'num_contactos_otras_campanas']

# 1.4 Numéricas sin transformar (pasan directas a escalado en FASE 3)
for c in ['edad', 'variacion_tasa_empleo', 'euribor3m']:
    df_f1[c] = df[c].astype(float)
cols_fase1_numericas += ['edad', 'variacion_tasa_empleo', 'euribor3m']
cols_intermedias_excluir += ['edad', 'variacion_tasa_empleo', 'euribor3m']

print('✅ FASE 1 OK -> features numéricas pre-escalado:', len(cols_fase1_numericas))

# ── FASE 2: Features binarias / cíclicas (NO escalar) ────────────────
df_f2 = pd.DataFrame(index=df.index)

# 2.1 OHE con drop='first' (obligatorio para evitar multicolinealidad)
CATS_OHE = ['trabajo', 'estado_civil', 'impago', 'prestamo_hipotecario',
            'prestamo_personal', 'canal_de_contacto', 'resultado_campana_anterior']
for col in CATS_OHE:
    ohe = OneHotEncoder(drop='first', sparse_output=False, dtype=np.int8)
    arr = ohe.fit_transform(df[[col]])
    cats_resto = ohe.categories_[0][1:]  # categorías tras eliminar la referencia
    for i, cat in enumerate(cats_resto):
        nombre = f'{col}_{cat}'
        df_f2[nombre] = arr[:, i]
    cols_intermedias_excluir.append(col)

cols_fase2_noescalar.extend(df_f2.columns)

# 2.2 Flag: contactado en campaña previa (num_dias >= 0)
df_f2['contactado_previamente'] = (df['num_dias_ultimo_contacto'] >= 0).astype(np.int8)
cols_fase2_noescalar.append('contactado_previamente')

# 2.3 mes: features cíclicas seno/coseno (ya en [-1,1], NO escalar)
mes_num = df['mes'].map(MAP_MES).astype(float)
angulo = 2 * np.pi * mes_num / 12
df_f2['mes_sin'] = np.sin(angulo)
df_f2['mes_cos'] = np.cos(angulo)
cols_fase2_noescalar += ['mes_sin', 'mes_cos']
cols_intermedias_excluir.append('mes')

print('✅ FASE 2 OK -> features binarias/cíclicas:', len(cols_fase2_noescalar))
print(f'   Total dummies OHE: {df_f2.shape[1] - 3} | flag: 1 | cíclicas: 2')
print(f'   Columnas pre-escalado FASE 1: {cols_fase1_numericas}')

✅ FASE 0 OK


✅ FASE 1 OK -> features numéricas pre-escalado: 7
✅ FASE 2 OK -> features binarias/cíclicas: 26
   Total dummies OHE: 23 | flag: 1 | cíclicas: 2
   Columnas pre-escalado FASE 1: ['formacion_oe_imp', 'num_dias_ultimo_contacto_imp', 'num_contactos_esta_campana_log', 'num_contactos_otras_campanas_log', 'edad', 'variacion_tasa_empleo', 'euribor3m']


In [9]:
# ── FASE 3: Escalado selectivo (StandardScaler solo sobre FASE 1) ────
ss = StandardScaler()
arr_ss = ss.fit_transform(df_f1[cols_fase1_numericas])
df_f3 = pd.DataFrame(arr_ss,
                     columns=[f'{c}_ss' for c in cols_fase1_numericas],
                     index=df.index)

# Las versiones pre-escalado pasan a ser intermedias (no van al df final)
cols_intermedias_excluir += cols_fase1_numericas
cols_finales_escaladas = list(df_f3.columns)

print('✅ FASE 3 OK -> features escaladas:', len(cols_finales_escaladas))

# ── FASE 4: Unión final (solo versiones finales + target) ────────────
df_final = pd.concat([df[[TARGET]], df_f2, df_f3], axis=1)

# VALIDACIONES OBLIGATORIAS
# 1) Nº de filas conservado
assert df_final.shape[0] == df.shape[0], 'ERROR: Pérdida de filas'
# 2) Target presente
assert TARGET in df_final.columns, f"ERROR: Target '{TARGET}' ausente"
# 3) Sin columnas intermedias ni originales transformadas
intrusas = set(df_final.columns).intersection(cols_intermedias_excluir)
assert len(intrusas) == 0, f'ERROR: Columnas intermedias en df final: {intrusas}'
# 4) Sin nombres duplicados
assert len(df_final.columns) == len(set(df_final.columns)), 'ERROR: Nombres duplicados'
# 5) Control de NaN
nan_tot = int(df_final.isnull().sum().sum())
print(f'   NaN totales en df_final: {nan_tot}')

# Renombrar a la variable estándar del proyecto
df = df_final.copy()

print('✅ FASE 4 OK -> df final listo')
print(f'   Filas: {df.shape[0]:,} | Columnas: {df.shape[1]}')
print(f'   Columnas finales ({len(df.columns)}):')
print(list(df.columns))

✅ FASE 3 OK -> features escaladas: 7
   NaN totales en df_final: 0
✅ FASE 4 OK -> df final listo
   Filas: 28,015 | Columnas: 34
   Columnas finales (34):
['contrata_fondos', 'trabajo_blue-collar', 'trabajo_entrepreneur', 'trabajo_housemaid', 'trabajo_management', 'trabajo_retired', 'trabajo_self-employed', 'trabajo_services', 'trabajo_student', 'trabajo_technician', 'trabajo_unemployed', 'trabajo_unknown', 'estado_civil_married', 'estado_civil_single', 'estado_civil_unknown', 'impago_unknown', 'impago_yes', 'prestamo_hipotecario_unknown', 'prestamo_hipotecario_yes', 'prestamo_personal_unknown', 'prestamo_personal_yes', 'canal_de_contacto_telephone', 'resultado_campana_anterior_nonexistent', 'resultado_campana_anterior_success', 'contactado_previamente', 'mes_sin', 'mes_cos', 'formacion_oe_imp_ss', 'num_dias_ultimo_contacto_imp_ss', 'num_contactos_esta_campana_log_ss', 'num_contactos_otras_campanas_log_ss', 'edad_ss', 'variacion_tasa_empleo_ss', 'euribor3m_ss']


In [10]:
# ── Guardado del tablón transformado ─────────────────────────────────
RUTA_SALIDA = ENTRENAMIENTO / '04_train_tablon_transformado.pkl'
df.to_pickle(RUTA_SALIDA)
print('✅ Tablón transformado guardado')
print(f'   Ruta: {RUTA_SALIDA}')

# ── Estructura final del dataframe (para copilot-instructions.md) ────
df.info()

✅ Tablón transformado guardado
   Ruta: c:\Users\Admin\Desktop\Desktop\CarpetaFisica\Data Detective\Ciencia_Datos_2\Caso_Agentes\02_datos\03_Entrenamiento\04_train_tablon_transformado.pkl
<class 'pandas.DataFrame'>
RangeIndex: 28015 entries, 0 to 28014
Data columns (total 34 columns):
 #   Column                                  Non-Null Count  Dtype  
---  ------                                  --------------  -----  
 0   contrata_fondos                         28015 non-null  int64  
 1   trabajo_blue-collar                     28015 non-null  int8   
 2   trabajo_entrepreneur                    28015 non-null  int8   
 3   trabajo_housemaid                       28015 non-null  int8   
 4   trabajo_management                      28015 non-null  int8   
 5   trabajo_retired                         28015 non-null  int8   
 6   trabajo_self-employed                   28015 non-null  int8   
 7   trabajo_services                        28015 non-null  int8   
 8   trabajo_student    